# APPlicatuMente — comparar modelos de Serena con GPU (Semana 6)

Reunión con el asesor (30 de octubre): probar otros modelos de lenguaje en un lugar con GPU gratis.
Este cuaderno corre el pipeline **real** de Serena (analizador, motor de Wang, guardia de salida) con varios modelos GGUF
sobre `bateria/conversaciones_s6.json` y guarda, por mensaje, la respuesta, el tiempo y lo que la guardia tuvo que descartar.

**Antes de correr**
1. En Kaggle: *Settings → Accelerator → GPU T4 x2* (o P100) y *Internet → On*.
2. Solo se usan frases inventadas por el equipo. **Nunca** subir aquí mensajes de estudiantes reales (son datos de menores).
3. **Código.** El repositorio ya no es público, así que el código se sube como *dataset* privado:
   - En la computadora, desde `mock_backend/`: `python -m bateria.paquete_kaggle --salida ~/aptm_kaggle_s6.zip`
   - En Kaggle: *Add Input → Upload → New Dataset*, sube el zip, ponle de nombre `aptm-codigo` y déjalo **Private**.
   - El zip no lleva datos de estudiantes ni la revisión clínica completa (solo id y estado de cada texto).
   - Si no hay dataset, el cuaderno intenta clonar la rama `prototipo/diario-semaforo` (solo funciona si el repositorio vuelve a ser público).

**Cómo leer el resultado**: `descartadas` = veces que la guardia tuvo que tirar la oración del modelo (usted, género, supone malestar, lista cortada…). Menos es mejor. La calidad se califica después a ciegas con las respuestas guardadas.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import glob, os, shutil, subprocess
DESTINO = "/kaggle/working/mock_backend"
encontrado = glob.glob("/kaggle/input/**/mock_backend/bateria/comparar_modelos.py", recursive=True)
if encontrado:
    origen = encontrado[0].rsplit("/bateria/", 1)[0]
    print("Código del dataset:", origen)
else:
    print("Sin dataset: se clona desde GitHub")
    subprocess.run(["git", "clone", "--depth", "1", "-b", "prototipo/diario-semaforo",
                    "https://github.com/opelo1209/APPlicatuMente.git", "/kaggle/working/repo"], check=True)
    origen = "/kaggle/working/repo/mock_backend"
shutil.copytree(origen, DESTINO, dirs_exist_ok=True)
os.chdir(DESTINO)
print(os.getcwd(), sorted(os.listdir("bateria"))[:5])

Dependencias. `llama-cpp-python` con CUDA: primero se intenta una rueda ya compilada; si no hay para esta versión de CUDA, se compila (tarda ~10–15 min).

In [ ]:
!pip install -q pysentimiento==0.7.3 huggingface_hub
!pip install -q llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
import llama_cpp
print(llama_cpp.__version__, 'GPU:', llama_cpp.llama_supports_gpu_offload())

In [ ]:
# Solo si la celda anterior imprimió GPU: False
import llama_cpp
if not llama_cpp.llama_supports_gpu_offload():
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir llama-cpp-python
    print('Reinicia el kernel (Run → Restart) y sigue desde la celda de modelos.')

Modelos a comparar (todos Q4_K_M). El primero es el que usa hoy el servidor. Se pueden agregar otros
(Llama 3.2 3B, Gemma 2 2B…) con su repositorio y archivo en Hugging Face; **verificar el nombre exacto del archivo** en la página del modelo.

In [ ]:
from huggingface_hub import hf_hub_download
MODELOS = [
    ("Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-q4_k_m.gguf"),   # el del servidor
    ("Qwen/Qwen2.5-3B-Instruct-GGUF",   "qwen2.5-3b-instruct-q4_k_m.gguf"),
    ("bartowski/Qwen2.5-7B-Instruct-GGUF", "Qwen2.5-7B-Instruct-Q4_K_M.gguf"),
]
rutas = [hf_hub_download(repo, archivo, local_dir="/kaggle/working/modelos") for repo, archivo in MODELOS]
rutas

In [ ]:
import os, subprocess
os.environ["SERENA_CAPAS_GPU"] = "-1"   # todas las capas en la GPU
os.environ["SERENA_HILOS"] = "4"
args = ["python", "-m", "bateria.comparar_modelos", "--salida", "/kaggle/working/comparacion_modelos.json"]
for r in rutas:
    args += ["--modelo", r]
print(subprocess.run(args, capture_output=True, text=True).stdout)

In [ ]:
import json, pandas as pd
d = json.load(open("/kaggle/working/comparacion_modelos.json"))
pd.DataFrame({m: v["resumen"] for m, v in d["modelos"].items()}).T

In [ ]:
# Respuestas lado a lado (para revisar y para la calificación a ciegas)
filas = []
for m, v in d["modelos"].items():
    for f in v["filas"]:
        filas.append({"modelo": m, "conv": f["conversacion"], "turno": f["turno"], "mensaje": f["mensaje"],
                      "respuesta": f["texto"], "guardia": f["guardia"], "s": f["segundos"]})
tabla = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", 200)
tabla.pivot_table(index=["conv", "turno", "mensaje"], columns="modelo", values="respuesta", aggfunc="first")

Descarga `/kaggle/working/comparacion_modelos.json` (panel *Output*) y guárdalo en `Semana_6/reporte_de_actividades/herramientas/metricas/`.